<a href="https://colab.research.google.com/github/raheelarif86/AI_Training_November25/blob/main/Final_OT_Cybersecurity_27_August_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG (Local Files)
# ============================================================
# This version:
# - Uses your 3 uploaded files from /data folder
# - Removes Gradio UI
# - Provides a single function run_full_assessment()
# - Fully commented and cleaned for Colab
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx

import os
import io
import json
import numpy as np
import pandas as pd
import httpx
import docx
from openai import OpenAI

# ---------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
import os
from google.colab import userdata
import httpx
from openai import OpenAI

# Load your secret key stored under the name "openai"
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

# Initialize OpenAI client
client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    """Reads .txt, .csv, .json, .pdf, .docx"""
    if not os.path.exists(path):
        return ""

    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent
# ============================================================

LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

def likelihood_agent(threat_actor, exposure, title, causes):
    tac = clamp_likelihood(threat_actor)
    exp = clamp_likelihood(exposure)

    vuln = infer_likelihood_from_model(title+" (vuln)", causes)
    hist = infer_likelihood_from_model(title+" (history)", causes)

    scores = {
        "threat_actor_capability": L2S[tac],
        "vulnerability_exploitability": L2S[vuln],
        "exposure": L2S[exp],
        "historical_occurrences": L2S[hist]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2L[scores[max_factor]]

    explanation_prompt = f"""
Explain in 4–6 sentences why the likelihood is '{overall}'.
Inputs:
TAC={tac}, VULN={vuln}, EXP={exp}, HIST={hist}.
Driving factor: {max_factor}.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    return overall, exp_resp.choices[0].message.content, {
        "threat_actor_capability": tac,
        "vulnerability_exploitability": vuln,
        "exposure": exp,
        "historical_occurrences": hist
    }

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):

    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood}
Likelihood basis:
{likelihood_basis}

Impact={impact}
Impact basis:
{impact_basis}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):

    compliance_path = "data/NEI_08-09_Rev_06.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/Nuclear_OT_Control_Library.docx"

    # 1) Likelihood
    L, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    # 2) Impact
    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    # 3) Risk Estimation
    R, R_expl = risk_estimator(L, I, heatmap_path)

    # 4) Controls
    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    # 5) Final Report
    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    return report

# ============================================================
# Example Run
# ============================================================

report = run_full_assessment(
    asset_category="Engineering Workstation",
    asset_criticality="High",
    risk_title="Unauthorized Remote Access",
    risk_causes="Weak authentication, exposed services",
    threat_actor_capability="likely",
    exposure="possible",
    safety="moderate",
    availability="major",
    confidentiality="moderate",
    integrity="major"
)

print(report)


# OT Nuclear Cyber Risk Report

## Asset Overview
- **Asset**: Engineering Workstation
- **Criticality**: High

## Risk Assessment
### Risk Title
- **Unauthorized Remote Access**

### Causes
- **Weak Authentication**
- **Exposed Services**

### Likelihood
- **Rating**: Likely
- **Likelihood Basis**:
  - **Threat Actor Capability (TAC)**: Rated as 'likely' due to the actors possessing the necessary skills and resources for effective attacks.
  - **Vulnerability (VULN)**: Rated as 'likely' indicating identifiable weaknesses in the system that can be exploited.
  - **Exposure (EXP)**: Rated as 'possible'; however, this does not significantly diminish the overall risk.
  - **History of Similar Incidents (HIST)**: Rated as 'likely', reinforcing the probability of future occurrences based on past successful attacks.

### Impact
- **Rating**: Major
- **Impact Basis**:
  - **Availability**: Major - Compromise can severely disrupt operations.
  - **Integrity**: Major - Breaches could undermine 

In [10]:
# ============================================================
# OT Nuclear Risk Assessment – Multi-Agent + RAG + Gradio UI
# ============================================================

!pip install openai==1.16.0 numpy pandas PyPDF2 python-docx openpyxl tiktoken httpx gradio==4.19.2

import os
import io
import json
import numpy as np
import pandas as pd
import httpx
import docx
import gradio as gr
from google.colab import userdata
from openai import OpenAI

# ---------------------------------
# 1) Configure OpenAI client using Colab Secrets
# ----------------------------------------------
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    http_client=httpx.Client()
)

CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

# ============================================================
# 2) Utility: Embeddings + RAG
# ============================================================

def get_embedding(text: str):
    text = text.replace("\n", " ")
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[text])
    return resp.data[0].embedding

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-10))

def read_text_from_file(path: str):
    if not os.path.exists(path):
        return ""

    if path.endswith(".txt") or "." not in path.split("/")[-1]:
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".csv"):
        df = pd.read_csv(path)
        return df.to_csv(index=False)

    if path.endswith(".json"):
        return open(path, "r", encoding="utf-8", errors="ignore").read()

    if path.endswith(".pdf"):
        import PyPDF2
        text = []
        reader = PyPDF2.PdfReader(path)
        for page in reader.pages:
            text.append(page.extract_text() or "")
        return "\n".join(text)

    if path.endswith(".docx"):
        document = docx.Document(path)
        return "\n".join([p.text for p in document.paragraphs])

    return ""

def build_rag_index(path: str):
    raw = read_text_from_file(path)
    if not raw:
        return []

    words = raw.split()
    chunk_size = 700
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    index = []
    for ch in chunks:
        index.append({"chunk": ch, "embedding": get_embedding(ch)})
    return index

def rag_search(index, query, top_k=3):
    if not index:
        return ""
    q_emb = get_embedding(query)
    scored = [(cosine_similarity(q_emb, e["embedding"]), e["chunk"]) for e in index]
    scored.sort(reverse=True, key=lambda x: x[0])
    return "\n\n".join([c for _, c in scored[:top_k]])

# ============================================================
# 3) Likelihood Calculator Agent
# ============================================================

LIKELIHOOD = ["very unlikely", "unlikely", "possible", "likely", "very likely"]
L2S = {lvl: i for i, lvl in enumerate(LIKELIHOOD)}
S2L = {i: lvl for i, lvl in enumerate(LIKELIHOOD)}

def clamp_likelihood(v):
    v = v.lower().strip()
    for lvl in LIKELIHOOD:
        if lvl in v:
            return lvl
    return "possible"

def infer_likelihood_from_model(title, causes):
    prompt = f"""
Map the likelihood to one of:
very unlikely, unlikely, possible, likely, very likely.

Risk title: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_likelihood(resp.choices[0].message.content)

def likelihood_agent(threat_actor, exposure, title, causes):
    tac = clamp_likelihood(threat_actor)
    exp = clamp_likelihood(exposure)

    vuln = infer_likelihood_from_model(title+" (vuln)", causes)
    hist = infer_likelihood_from_model(title+" (history)", causes)

    scores = {
        "threat_actor_capability": L2S[tac],
        "vulnerability_exploitability": L2S[vuln],
        "exposure": L2S[exp],
        "historical_occurrences": L2S[hist]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2L[scores[max_factor]]

    explanation_prompt = f"""
Explain in 4–6 sentences why the likelihood is '{overall}'.
Inputs:
TAC={tac}, VULN={vuln}, EXP={exp}, HIST={hist}.
Driving factor: {max_factor}.
"""
    exp_resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":explanation_prompt}],
        temperature=0.5
    )

    return overall, exp_resp.choices[0].message.content, {
        "threat_actor_capability": tac,
        "vulnerability_exploitability": vuln,
        "exposure": exp,
        "historical_occurrences": hist
    }

# ============================================================
# 4) Impact Calculator Agent
# ============================================================

IMPACT = ["negligible","marginal","moderate","major","severe"]
I2S = {lvl:i for i,lvl in enumerate(IMPACT)}
S2I = {i:lvl for i,lvl in enumerate(IMPACT)}

def clamp_impact(v):
    v = v.lower().strip()
    for lvl in IMPACT:
        if lvl in v:
            return lvl
    return "moderate"

def infer_compliance_impact(rag_index, title, causes):
    ctx = rag_search(rag_index, f"Compliance impact for {title} {causes}", 3)
    prompt = f"""
Given compliance context:

{ctx}

Map compliance impact to:
negligible, marginal, moderate, major, severe.

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )
    return clamp_impact(resp.choices[0].message.content)

def infer_reputation(asset_category, title, causes):
    prompt = f"""
Map reputation impact to:
negligible, marginal, moderate, major, severe.

Asset: {asset_category}
Risk: {title}
Causes: {causes}

Output ONLY the label.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return clamp_impact(resp.choices[0].message.content)

def impact_agent(asset_category, safety, availability, confidentiality, integrity, compliance_path, title, causes):
    s = clamp_impact(safety)
    a = clamp_impact(availability)
    c = clamp_impact(confidentiality)
    i = clamp_impact(integrity)

    comp_index = build_rag_index(compliance_path)
    comp = infer_compliance_impact(comp_index, title, causes)
    rep = infer_reputation(asset_category, title, causes)

    scores = {
        "safety": I2S[s],
        "availability": I2S[a],
        "confidentiality": I2S[c],
        "integrity": I2S[i],
        "compliance": I2S[comp],
        "reputation": I2S[rep]
    }

    max_factor = max(scores, key=scores.get)
    overall = S2I[scores[max_factor]]

    prompt = f"""
Explain in 4–7 sentences why impact = '{overall}'.
Factors:
Safety={s}, Availability={a}, Confidentiality={c}, Integrity={i},
Compliance={comp}, Reputation={rep}.
Driving factor: {max_factor}.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.5
    )

    return overall, resp.choices[0].message.content, {
        "safety": s,
        "availability": a,
        "confidentiality": c,
        "integrity": i,
        "compliance": comp,
        "reputation": rep
    }

# ============================================================
# 5) Risk Estimator (Heatmap)
# ============================================================

def risk_estimator(likelihood, impact, heatmap_path):
    df = pd.read_excel(heatmap_path)
    df.columns = [c.lower().strip() for c in df.columns]
    df[df.columns[0]] = df[df.columns[0]].str.lower().str.strip()
    df = df.set_index(df.columns[0])

    like = likelihood.lower()
    imp = impact.lower()

    rating = df.loc[like, imp]

    prompt = f"""
Explain in 3–5 sentences why likelihood={likelihood} and impact={impact}
produce risk rating '{rating}' in a nuclear OT heatmap.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )

    return rating, resp.choices[0].message.content

# ============================================================
# 6) Response Planner (Control Library RAG)
# ============================================================

def response_planner(risk_rating, title, causes, asset_category, asset_criticality, control_path):
    index = build_rag_index(control_path)
    ctx = rag_search(index, f"{risk_rating} {title} {causes}", 5)

    prompt = f"""
Control library context:

{ctx}

Risk rating: {risk_rating}
Asset: {asset_category}, Criticality: {asset_criticality}

Provide:
- Recommended controls (bulleted)
- Implementation priorities (short paragraphs)
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.4
    )
    controls = resp.choices[0].message.content

    rationale_prompt = f"""
Explain in 4–6 sentences the rationale for the above controls
for risk '{title}' rated '{risk_rating}'.
"""
    r2 = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":rationale_prompt}],
        temperature=0.4
    )

    return controls, r2.choices[0].message.content

# ============================================================
# 7) Final Report Generator
# ============================================================

def generate_report(asset_category, asset_criticality, title, causes,
                    likelihood, likelihood_basis, likelihood_details,
                    impact, impact_basis, impact_details,
                    rating, rating_expl,
                    controls, controls_rationale):

    prompt = f"""
Generate a structured OT Nuclear Cyber Risk Report.

Context:
Asset={asset_category}, Criticality={asset_criticality}
Risk title={title}
Causes={causes}

Likelihood={likelihood}
Likelihood basis:
{likelihood_basis}

Impact={impact}
Impact basis:
{impact_basis}

Risk rating={rating}
Rating explanation:
{rating_expl}

Controls:
{controls}

Rationale:
{controls_rationale}

Write professionally with headings and bullet points.
"""
    resp = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role":"user","content":prompt}],
        temperature=0.3
    )
    return resp.choices[0].message.content

# ============================================================
# 8) MAIN FUNCTION TO RUN EVERYTHING
# ============================================================

def run_full_assessment(
    asset_category,
    asset_criticality,
    risk_title,
    risk_causes,
    threat_actor_capability,
    exposure,
    safety,
    availability,
    confidentiality,
    integrity
):

    compliance_path = "data/Compliance.pdf"
    heatmap_path = "data/Nuclear_OT_Risk_Heatmap.xlsx"
    control_path = "data/Control_Library.pdf"

    L, L_basis, L_details = likelihood_agent(
        threat_actor_capability, exposure, risk_title, risk_causes
    )

    I, I_basis, I_details = impact_agent(
        asset_category, safety, availability, confidentiality, integrity,
        compliance_path, risk_title, risk_causes
    )

    R, R_expl = risk_estimator(L, I, heatmap_path)

    C, C_rat = response_planner(
        R, risk_title, risk_causes, asset_category, asset_criticality, control_path
    )

    report = generate_report(
        asset_category, asset_criticality, risk_title, risk_causes,
        L, L_basis, L_details,
        I, I_basis, I_details,
        R, R_expl,
        C, C_rat
    )

    return (
        f"### Likelihood\n{L}\n\n{L_basis}",
        f"### Impact\n{I}\n\n{I_basis}",
        f"### Risk Rating\n{R}\n\n{R_expl}",
        f"### Controls\n{C}\n\n### Rationale\n{C_rat}",
        report
    )

# ============================================================
# 9) GRADIO UI
# ============================================================

with gr.Blocks(title="OT Nuclear Risk Assessment") as ui:

    gr.Markdown("# 🔐 OT Nuclear Cyber Risk Assessment Tool")
    gr.Markdown("Provide inputs below to generate a full multi‑agent risk assessment.")

    with gr.Row():
        asset_category = gr.Dropdown(
            ["Operator Workstation","Engineering Workstation","Server","Network Switch","Firewall"],
            label="Asset Category"
        )
        asset_criticality = gr.Dropdown(
            ["Very High","High","Medium","Low"],
            label="Asset Criticality"
        )

    risk_title = gr.Textbox(label="Risk Title")
    risk_causes = gr.Textbox(label="Risk Causes", lines=3)

    with gr.Row():
        threat_actor_capability = gr.Dropdown(LIKELIHOOD, label="Threat Actor Capability")
        exposure = gr.Dropdown(LIKELIHOOD, label="Exposure")

    with gr.Row():
        safety = gr.Dropdown(IMPACT, label="Safety Impact")
        availability = gr.Dropdown(IMPACT, label="Availability Impact")
        confidentiality = gr.Dropdown(IMPACT, label="Confidentiality Impact")
        integrity = gr.Dropdown(IMPACT, label="Integrity Impact")

    run_btn = gr.Button("Run Assessment")

    likelihood_out = gr.Markdown()
    impact_out = gr.Markdown()
    risk_out = gr.Markdown()
    controls_out = gr.Markdown()
    report_out = gr.Markdown()

    run_btn.click(
        run_full_assessment,
        inputs=[
            asset_category,
            asset_criticality,
            risk_title,
            risk_causes,
            threat_actor_capability,
            exposure,
            safety,
            availability,
            confidentiality,
            integrity
        ],
        outputs=[
            likelihood_out,
            impact_out,
            risk_out,
            controls_out,
            report_out
        ]
    )

ui.launch(share=True)


IMPORTANT: You are using gradio version 4.19.2, however version 4.44.1 is available, please upgrade.
--------
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://eae905c3e036d7a250.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
